# Experiments & inference: practice notebook

5 exercises, each with hidden tests. Data is loaded as **polars**; pandas only appears where a library insists on it.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("i3")` prints the reference solution.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, scipy, statsmodels, scikit-learn, matplotlib)
import importlib.util, subprocess, sys
for _p in ['polars']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The data every exercise uses. Read it if you like; don't change it.

In [ ]:
import numpy as np
import polars as pl
from scipy import stats
_g = np.random.default_rng(7)
ab = pl.DataFrame({"group": np.repeat(["control", "treatment"], 5000),
                   "converted": np.concatenate([_g.binomial(1, 0.100, 5000), _g.binomial(1, 0.115, 5000)])})
baskets = np.round(_g.lognormal(3.5, 0.9, 400), 2)

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJpMyI6IHsic29sdXRpb24iOiAiZGVmIGFhX2ZwcihuX3Rlc3RzPTQwMDAsIG49MjAwMCwgcD0wLjEsIHNlZWQ9MCk6XG4gICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpXG4gICAgYSA9IHJuZy5iaW5vbWlhbChuLCBwLCBzaXplPW5fdGVzdHMpXG4gICAgYiA9IHJuZy5iaW5vbWlhbChuLCBwLCBzaXplPW5fdGVzdHMpXG4gICAgcG9vbCA9IChhICsgYikgLyAoMiAqIG4pXG4gICAgc2UgPSBucC5zcXJ0KHBvb2wgKiAoMSAtIHBvb2wpICogMiAvIG4pXG4gICAgeiA9IChiIC0gYSkgLyBuIC8gc2VcbiAgICByZXR1cm4gZmxvYXQobnAubWVhbigyICogc3RhdHMubm9ybS5zZihucC5hYnMoeikpIDwgMC4wNSkpXG4iLCAidGVzdHMiOiAiXG5kZWYgY2xvc2UoYSwgYiwgcmVsPTFlLTYsIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGEgPSBmbG9hdChhKTsgYiA9IGZsb2F0KGIpXG4gICAgYXNzZXJ0IGFicyhhIC0gYikgPD0gcmVsICogbWF4KDEuMCwgYWJzKGIpKSwgZlwie3doYXR9ID0ge2E6LjZnfSwgZXhwZWN0ZWQge2I6LjZnfVwiXG5kZWYgYmV0d2VlbihhLCBsbywgaGksIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGFzc2VydCBsbyA8PSBmbG9hdChhKSA8PSBoaSwgZlwie3doYXR9ID0ge2Zsb2F0KGEpOi40Zn0sIGV4cGVjdGVkIGJldHdlZW4ge2xvfSBhbmQge2hpfVwiXG5kZWYgc2FtZV9rZXlzKGEsIGIsIHdoYXQ9XCJyZXN1bHRcIik6XG4gICAgYXNzZXJ0IGlzaW5zdGFuY2UoYSwgZGljdCksIGZcInt3aGF0fSBzaG91bGQgYmUgYSBkaWN0LCBnb3Qge3R5cGUoYSkuX19uYW1lX199XCJcbiAgICBhc3NlcnQgc2V0KGEpID09IHNldChiKSwgZlwie3doYXR9IGtleXMge3NvcnRlZChhKX0sIGV4cGVjdGVkIHtzb3J0ZWQoYil9XCJcblxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuXG5URVNUUyA9IFtcbiAgKFwiYWJvdXQgNSUgd2l0aCBkZWZhdWx0IHNldHRpbmdzXCIsIGxhbWJkYTogYmV0d2VlbihVW1wiYWFfZnByXCJdKCksIDAuMDM1LCAwLjA2NSwgXCJmYWxzZS1wb3NpdGl2ZSByYXRlXCIpKSxcbiAgKFwiYWJvdXQgNSUgd2l0aCBhbm90aGVyIHNlZWRcIiwgbGFtYmRhOiBiZXR3ZWVuKFVbXCJhYV9mcHJcIl0oc2VlZD03LCBuPTUwMDAsIHA9MC4yKSwgMC4wMzUsIDAuMDY1LCBcImZhbHNlLXBvc2l0aXZlIHJhdGVcIikpLFxuXSIsICJsYW5nIjogInB5In0sICJpNSI6IHsic29sdXRpb24iOiAiZGVmIHBlZWtfZnByKG5fc2ltcz0yMDAwLCBkYXlzPTIwLCBuX3Blcl9kYXk9NTAwLCBwPTAuMSwgc2VlZD0wKTpcbiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZClcbiAgICBhID0gcm5nLmJpbm9taWFsKG5fcGVyX2RheSwgcCwgc2l6ZT0obl9zaW1zLCBkYXlzKSkuY3Vtc3VtKGF4aXM9MSlcbiAgICBiID0gcm5nLmJpbm9taWFsKG5fcGVyX2RheSwgcCwgc2l6ZT0obl9zaW1zLCBkYXlzKSkuY3Vtc3VtKGF4aXM9MSlcbiAgICBuID0gbl9wZXJfZGF5ICogbnAuYXJhbmdlKDEsIGRheXMgKyAxKVxuICAgIHBvb2wgPSAoYSArIGIpIC8gKDIgKiBuKVxuICAgIHNlID0gbnAuc3FydChwb29sICogKDEgLSBwb29sKSAqIDIgLyBuKVxuICAgIHogPSAoYiAtIGEpIC8gbiAvIHNlXG4gICAgc2lnID0gMiAqIHN0YXRzLm5vcm0uc2YobnAuYWJzKHopKSA8IDAuMDVcbiAgICByZXR1cm4gZmxvYXQoc2lnLmFueShheGlzPTEpLm1lYW4oKSlcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgY2xvc2UoYSwgYiwgcmVsPTFlLTYsIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGEgPSBmbG9hdChhKTsgYiA9IGZsb2F0KGIpXG4gICAgYXNzZXJ0IGFicyhhIC0gYikgPD0gcmVsICogbWF4KDEuMCwgYWJzKGIpKSwgZlwie3doYXR9ID0ge2E6LjZnfSwgZXhwZWN0ZWQge2I6LjZnfVwiXG5kZWYgYmV0d2VlbihhLCBsbywgaGksIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGFzc2VydCBsbyA8PSBmbG9hdChhKSA8PSBoaSwgZlwie3doYXR9ID0ge2Zsb2F0KGEpOi40Zn0sIGV4cGVjdGVkIGJldHdlZW4ge2xvfSBhbmQge2hpfVwiXG5cblRFU1RTID0gW1xuICAoXCJvbmUgbG9vayBpcyBhYm91dCA1JVwiLCBsYW1iZGE6IGJldHdlZW4oVVtcInBlZWtfZnByXCJdKGRheXM9MSksIDAuMDM1LCAwLjA3LCBcInJhdGUgd2l0aCAxIGxvb2tcIikpLFxuICAoXCIyMCBkYWlseSBsb29rcyBpbmZsYXRlIGl0XCIsIGxhbWJkYTogYmV0d2VlbihVW1wicGVla19mcHJcIl0oKSwgMC4xNywgMC4zNSwgXCJyYXRlIHdpdGggMjAgbG9va3NcIikpLFxuXSIsICJsYW5nIjogInB5In0sICJpNiI6IHsic29sdXRpb24iOiAiZnJvbSBzdGF0c21vZGVscy5zdGF0cy5wcm9wb3J0aW9uIGltcG9ydCBwcm9wb3J0aW9uc196dGVzdFxuXG5kZWYgYWJfenRlc3QoZGYpOlxuICAgIHMgPSBkZi5ncm91cF9ieShcImdyb3VwXCIpLmFnZyhwbC5jb2woXCJjb252ZXJ0ZWRcIikuc3VtKCkuYWxpYXMoXCJ4XCIpLCBwbC5sZW4oKS5hbGlhcyhcIm5cIikpXG4gICAgdCA9IHMuZmlsdGVyKHBsLmNvbChcImdyb3VwXCIpID09IFwidHJlYXRtZW50XCIpXG4gICAgYyA9IHMuZmlsdGVyKHBsLmNvbChcImdyb3VwXCIpID09IFwiY29udHJvbFwiKVxuICAgIHosIHAgPSBwcm9wb3J0aW9uc196dGVzdChbdFtcInhcIl0uaXRlbSgpLCBjW1wieFwiXS5pdGVtKCldLCBbdFtcIm5cIl0uaXRlbSgpLCBjW1wiblwiXS5pdGVtKCldKVxuICAgIHJldHVybiBmbG9hdCh6KSwgZmxvYXQocClcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoZGYpOlxuICAgIGEgPSBVW1wiYWJfenRlc3RcIl0oZGYpOyBiID0gUltcImFiX3p0ZXN0XCJdKGRmKVxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lIGFuZCBsZW4oYSkgPT0gMiwgXCJyZXR1cm4gKHosIHBfdmFsdWUpXCJcbiAgICBjbG9zZShhWzBdLCBiWzBdLCAxZS02LCBcInpcIik7IGNsb3NlKGFbMV0sIGJbMV0sIDFlLTYsIFwicC12YWx1ZVwiKVxuVEVTVFMgPSBbKFwiYWJcIiwgbGFtYmRhOiBfdChhYikpLCAoXCJoaWRkZW46IHVuZXF1YWwgZ3JvdXAgc2l6ZXNcIiwgbGFtYmRhOiBfdChhYi5zbGljZSgxMDAwLCA3MDAwKSkpXVxuIiwgImxhbmciOiAicHkifSwgImk3IjogeyJzb2x1dGlvbiI6ICJpbXBvcnQgbWF0aFxuZnJvbSBzdGF0c21vZGVscy5zdGF0cy5wb3dlciBpbXBvcnQgTm9ybWFsSW5kUG93ZXJcbmZyb20gc3RhdHNtb2RlbHMuc3RhdHMucHJvcG9ydGlvbiBpbXBvcnQgcHJvcG9ydGlvbl9lZmZlY3RzaXplXG5cbmRlZiBuX3Blcl9ncm91cChwMCwgcDEsIGFscGhhPTAuMDUsIHBvd2VyPTAuOCk6XG4gICAgaCA9IGFicyhwcm9wb3J0aW9uX2VmZmVjdHNpemUocDEsIHAwKSlcbiAgICBuID0gTm9ybWFsSW5kUG93ZXIoKS5zb2x2ZV9wb3dlcihlZmZlY3Rfc2l6ZT1oLCBhbHBoYT1hbHBoYSwgcG93ZXI9cG93ZXIsIHJhdGlvPTEuMCwgYWx0ZXJuYXRpdmU9XCJ0d28tc2lkZWRcIilcbiAgICByZXR1cm4gaW50KG1hdGguY2VpbChuKSlcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoKmFyZ3MsICoqa3cpOlxuICAgIGEgPSBVW1wibl9wZXJfZ3JvdXBcIl0oKmFyZ3MsICoqa3cpOyBiID0gUltcIm5fcGVyX2dyb3VwXCJdKCphcmdzLCAqKmt3KVxuICAgIGFzc2VydCBhID09IGIsIGZcIm4gPSB7YX0sIGV4cGVjdGVkIHtifVwiXG5URVNUUyA9IFsoXCIxMCUgXHUyMTkyIDExLjUlXCIsIGxhbWJkYTogX3QoMC4xMCwgMC4xMTUpKSwgKFwiaGlkZGVuOiA1JSBcdTIxOTIgNiUsIHBvd2VyIDAuOVwiLCBsYW1iZGE6IF90KDAuMDUsIDAuMDYsIHBvd2VyPTAuOSkpLFxuICAgICAgICAgKFwiaGlkZGVuOiBhIGRyb3AsIDIwJSBcdTIxOTIgMTglXCIsIGxhbWJkYTogX3QoMC4yMCwgMC4xOCkpXVxuIiwgImxhbmciOiAicHkifSwgImk4IjogeyJzb2x1dGlvbiI6ICJkZWYgbWVkaWFuX2NpKHgsIHNlZWQ9MCk6XG4gICAga3cgPSBkaWN0KG5fcmVzYW1wbGVzPTk5OTksIGNvbmZpZGVuY2VfbGV2ZWw9MC45NSwgbWV0aG9kPVwicGVyY2VudGlsZVwiKVxuICAgIHRyeTpcbiAgICAgICAgcmVzID0gc3RhdHMuYm9vdHN0cmFwKCh4LCksIG5wLm1lZGlhbiwgcm5nPW5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKSwgKiprdylcbiAgICBleGNlcHQgVHlwZUVycm9yOiAgIyBvbGRlciBzY2lweVxuICAgICAgICByZXMgPSBzdGF0cy5ib290c3RyYXAoKHgsKSwgbnAubWVkaWFuLCByYW5kb21fc3RhdGU9bnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpLCAqKmt3KVxuICAgIHJldHVybiBmbG9hdChyZXMuY29uZmlkZW5jZV9pbnRlcnZhbC5sb3cpLCBmbG9hdChyZXMuY29uZmlkZW5jZV9pbnRlcnZhbC5oaWdoKVxuXG5kZWYgbWVhbl9jaSh4KTpcbiAgICBtLCBzID0gbnAubWVhbih4KSwgbnAuc3RkKHgsIGRkb2Y9MSlcbiAgICBoID0gc3RhdHMubm9ybS5wcGYoMC45NzUpICogcyAvIG5wLnNxcnQobGVuKHgpKVxuICAgIHJldHVybiBmbG9hdChtIC0gaCksIGZsb2F0KG0gKyBoKVxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuZGVmIHNhbWVfa2V5cyhhLCBiLCB3aGF0PVwicmVzdWx0XCIpOlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIGRpY3QpLCBmXCJ7d2hhdH0gc2hvdWxkIGJlIGEgZGljdCwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IHNldChhKSA9PSBzZXQoYiksIGZcInt3aGF0fSBrZXlzIHtzb3J0ZWQoYSl9LCBleHBlY3RlZCB7c29ydGVkKGIpfVwiXG5cbmRlZiBfdCh4KTpcbiAgICBhID0gVVtcIm1lZGlhbl9jaVwiXSh4KTsgYiA9IFJbXCJtZWRpYW5fY2lcIl0oeClcbiAgICBjbG9zZShhWzBdLCBiWzBdLCAwLjAzLCBcIm1lZGlhbiBDSSBsb3dcIik7IGNsb3NlKGFbMV0sIGJbMV0sIDAuMDMsIFwibWVkaWFuIENJIGhpZ2hcIilcbiAgICBhID0gVVtcIm1lYW5fY2lcIl0oeCk7IGIgPSBSW1wibWVhbl9jaVwiXSh4KVxuICAgIGNsb3NlKGFbMF0sIGJbMF0sIDFlLTYsIFwibWVhbiBDSSBsb3dcIik7IGNsb3NlKGFbMV0sIGJbMV0sIDFlLTYsIFwibWVhbiBDSSBoaWdoXCIpXG5URVNUUyA9IFsoXCJiYXNrZXRzXCIsIGxhbWJkYTogX3QoYmFza2V0cykpLCAoXCJoaWRkZW46IGZpcnN0IDE1MCBiYXNrZXRzXCIsIGxhbWJkYTogX3QoYmFza2V0c1s6MTUwXSkpXVxuIiwgImxhbmciOiAicHkifX0=").decode())
_SETUP = _b64.b64decode("aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwb2xhcnMgYXMgcGwKZnJvbSBzY2lweSBpbXBvcnQgc3RhdHMKX2cgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoNykKYWIgPSBwbC5EYXRhRnJhbWUoeyJncm91cCI6IG5wLnJlcGVhdChbImNvbnRyb2wiLCAidHJlYXRtZW50Il0sIDUwMDApLAogICAgICAgICAgICAgICAgICAgImNvbnZlcnRlZCI6IG5wLmNvbmNhdGVuYXRlKFtfZy5iaW5vbWlhbCgxLCAwLjEwMCwgNTAwMCksIF9nLmJpbm9taWFsKDEsIDAuMTE1LCA1MDAwKV0pfSkKYmFza2V0cyA9IG5wLnJvdW5kKF9nLmxvZ25vcm1hbCgzLjUsIDAuOSwgNDAwKSwgMik=").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. Simulate A/A tests
*Easy · id `i3`*

<p>In an A/A test both groups are identical, so a test at α = 0.05 should flag about 5% of runs.</p>
<p>Write <code>aa_fpr(n_tests=4000, n=2000, p=0.1, seed=0)</code>: simulate <code>n_tests</code> A/A experiments with <code>n</code> users per group and true conversion <code>p</code>, run a two-sided pooled z-test on each, and return the fraction with p &lt; 0.05. Use <code>np.random.default_rng(seed)</code>.</p>
<p class=hint>Vectorise: draw all conversion counts at once with <code>rng.binomial(n, p, size=n_tests)</code>. Then try n = 50, p = 0.01 and see what breaks.</p>

In [ ]:
def aa_fpr(n_tests=4000, n=2000, p=0.1, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    print(aa_fpr())
    print(aa_fpr(n=200, p=0.03))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("i3")

---
## 2. The cost of peeking
*Medium · id `i5`*

<p>An analyst checks an A/A test every day and stops the first day p &lt; 0.05.</p>
<p>Write <code>peek_fpr(n_sims=2000, days=20, n_per_day=500, p=0.1, seed=0)</code>: each day both groups get <code>n_per_day</code> new users with true conversion <code>p</code>; after each day run the pooled z-test on the <b>cumulative</b> data. Return the fraction of simulations that were ever significant.</p>
<p class=hint><code>cumsum</code> over a (n_sims, days) array of daily counts avoids a loop. With <code>days=1</code> you should get back about 5%.</p>

In [ ]:
def peek_fpr(n_sims=2000, days=20, n_per_day=500, p=0.1, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    for d in (1, 5, 20, 60):
        print(d, peek_fpr(days=d))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("i5")

---
## 3. Same test with statsmodels proportions_ztest; check your hand result
*Easy · id `i6`*

<p><code>ab</code> (polars) has a <code>group</code> column (<code>control</code> / <code>treatment</code>) and a 0/1 <code>converted</code> column.</p>
<p>First compute the pooled two-proportion z statistic by hand (as in OpenIntro §6.2). Then write <code>ab_ztest(df)</code>: aggregate with <code>group_by</code>, call <code>proportions_ztest</code> with <b>treatment first</b>, and return <code>(z, p_value)</code> as floats. They should match your hand result.</p>

**Theory to know cold**
<ul><li>Pooled SE under H₀: √(p̂(1−p̂)(1/n₁ + 1/n₂)), p̂ = total conversions / total users.</li><li><code>proportions_ztest(count, nobs)</code> uses the pooled SE by default and tests count[0]/nobs[0] − count[1]/nobs[1].</li></ul>

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

def ab_ztest(df: pl.DataFrame):
    # your code here
    pass

# Sample run
try:
    print(ab.group_by("group").agg(pl.col("converted").mean()))
    print(ab_ztest(ab))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("i6")

---
## 4. statsmodels NormalIndPower: solve for n and compare
*Easy · id `i7`*

<p>Write <code>n_per_group(p0, p1, alpha=0.05, power=0.8)</code>: the users needed per group to detect a change from conversion <code>p0</code> to <code>p1</code> with a two-sided test. Use <code>proportion_effectsize</code> (Cohen's h) and <code>NormalIndPower().solve_power</code>, and round <b>up</b> to an integer.</p>
<p>Compare with the rule of thumb n ≈ 16 p(1−p) / Δ².</p>

**Theory to know cold**
<ul><li>n scales with 1/Δ²: halving the effect you want to detect quadruples the sample.</li><li>Cohen's h = 2 arcsin√p₁ − 2 arcsin√p₀ stabilises the variance of a proportion.</li></ul>

In [ ]:
import math
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

def n_per_group(p0, p1, alpha=0.05, power=0.8):
    # your code here
    pass

# Sample run
try:
    print(n_per_group(0.10, 0.115))
    print(16 * 0.1 * 0.9 / 0.015**2)
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("i7")

---
## 5. scipy.stats.bootstrap: CI for a median, compared with the normal CI for the mean
*Medium · id `i8`*

<p><code>baskets</code> is a skewed array of 400 basket values.</p>
<p>Write <code>median_ci(x, seed=0)</code>: a 95% <b>percentile</b> bootstrap CI for the median with <code>scipy.stats.bootstrap</code> and 9,999 resamples, returned as <code>(low, high)</code>. Pass a generator <code>np.random.default_rng(seed)</code> (the argument is <code>rng=</code> in recent scipy, <code>random_state=</code> in older versions).</p>
<p>Write <code>mean_ci(x)</code>: the normal-approximation 95% CI x̄ ± z₀.₉₇₅ · s/√n (s with ddof = 1).</p>
<p>Why is the median's interval not centred on the sample median?</p>

**Theory to know cold**
<ul><li>The bootstrap treats the sample as the population and resamples it with replacement.</li><li>Percentile intervals are simple; BCa (the scipy default) corrects bias and skew.</li><li>For heavy-tailed data, the median is more stable than the mean.</li></ul>

In [ ]:
def median_ci(x, seed=0):
    # your code here
    pass

def mean_ci(x):
    # your code here
    pass

# Sample run
try:
    print("median", np.median(baskets), median_ci(baskets))
    print("mean", baskets.mean(), mean_ci(baskets))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("i8")